### Programming for Biomedical Informatics
#### Week 10 - Building & Analysing a Patient Similarity Network using Multi-modal Data Fusion

In this notebook we are going to use similarity network fusion (SNF) to integrate gene expression and DNA methylation data from individuals in the TCGA LUAD dataset that we used previously. This is a dataset for lung cancer and our meta-data includes information about smoking status, cancer status, gender, and ethnicity.

Our approach will first be to use Lasso regression against the smoking level (cigarettes_per_day) for an individual to identify features that are informative - this is one of many possible ways to reduce the number of features in our analysis.

Once we have reduced the number of features for both modalities we will build independent patient similarity networks (PSNs) from them.

Next we will use SNF to fuse the networks and maximise patient retention in the data (different numbers of patients have data for each modality)

Once fused we will sparsify the network to retain only the strongest edges and then perform clustering to see what sub-groups we find and how they relate to cancer status

The notebook is designed to give you an idea of how you might approach such an integrative approach we've made some assumptions and simplifying decisions along the way to make this tractable for the time we have in a session.

The data files you need for this notebook can be downloaded [here](https://datasync.ed.ac.uk/index.php/s/epDk1uiyWnz9UNF) with password 'pbi2025', put this data in a folder called 'data' that's located in the same folder as the notebook.

In [1]:
# import libraries
import pickle as pk
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
import snf
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Import the DNA Methylation Data
## Reminder of what DNA Methylation data:
# Permanent modificaiton made to DNA at various Cytosine residue at CG pair at cytogene, quite unlikely at ACAT. Marks structure. Regulates expresssion of genes.
with open("data/ISMB_TCGA_DNAm.pkl", "rb") as f:
    dnaMet = pk.load(f)

# Extract the Methylation Data
metData = dnaMet['datExpr']
metPatients, metFeatures = metData.shape
print(f'We have methylation data for',metPatients,'patients with',metFeatures,'features')

# Extract the MetaData
metMeta = dnaMet['datMeta']
metMeta.head()

We have methylation data for 459 patients with 300000 features


,patient,race,gender,sample_type,cigarettes_per_day,Smoked
rownames,,,,,,
TCGA-55-7914,TCGA-55-7914,white,female,Primary Tumor,0.273973,Smoker
TCGA-38-4631,TCGA-38-4631,white,female,Primary Tumor,2.191781,Smoker
TCGA-73-4658,TCGA-73-4658,white,female,Primary Tumor,1.369863,Smoker
TCGA-50-5932,TCGA-50-5932,white,male,Primary Tumor,0.000000,Never
TCGA-55-7576,TCGA-55-7576,black or african american,male,Primary Tumor,0.000000,Never


In [3]:
# Import the Gene Expression Data
with open("data/ISMB_TCGA_GE.pkl", "rb") as f:
    dnaExp = pk.load(f)

# Extract the Gene Expression Data
expData = dnaExp['datExpr']
expPatients, expFeatures = expData.shape
print(f'We have gene expression data for',expPatients,'patients with',expFeatures,'features')

# Extract the MetaData
expMeta = dnaExp['datMeta']
expMeta.head()

We have gene expression data for 498 patients with 22637 features


,patient,race,gender,sample_type,cigarettes_per_day,Smoked,sizeFactor,replaceable
_row,,,,,,,,
TCGA-38-7271,TCGA-38-7271,white,female,Primary Tumor,1.3699,Smoker,0.5841,True
TCGA-55-7914,TCGA-55-7914,white,female,Primary Tumor,0.274,Smoker,0.9873,True
TCGA-95-7043,TCGA-95-7043,white,female,Primary Tumor,2.1918,Smoker,0.5439,True
TCGA-73-4658,TCGA-73-4658,white,female,Primary Tumor,1.3699,Smoker,0.7715,True
TCGA-86-8076,TCGA-86-8076,white,male,Primary Tumor,0,Never,1.313,True


In [4]:
# Feature Selection using Lasso Regression

# regress the DNAMet features against the cigarettes per day
# Basic penalized regression: DNAm features -> cigarettes_per_day

# Response variable
y = metMeta['cigarettes_per_day']

# Drop samples with missing response
valid_idx = y.dropna().index
y = y.loc[valid_idx]

# Align methylation matrix rows to valid samples
X = metData.loc[valid_idx]

# Optional: remove features with too low variance (speeds up)
var_filter = VarianceThreshold(threshold=0.0005)  # adjust threshold as needed

# Build pipeline: variance filter -> scaling -> LassoCV
pipe = Pipeline([
    ('var', var_filter),
    ('scale', StandardScaler(with_mean=False)),  # with_mean=False for sparse-like large matrices
    ('lasso', LassoCV(cv=5, n_jobs=-1, max_iter=5000, verbose=False))
])

# Fit model
pipe.fit(X, y)

lasso = pipe.named_steps['lasso']
filtered_feature_names = X.columns[var_filter.get_support()]

# Extract non-zero coefficients
coef = lasso.coef_
nonzero_mask = coef != 0
selected_features = filtered_feature_names[nonzero_mask]
selected_coefs = coef[nonzero_mask]

# Report
print(f"Total original features: {X.shape[1]}")
print(f"Features after variance filter: {len(filtered_feature_names)}")
print(f"Non-zero (selected) features: {len(selected_features)}")

# Build results DataFrame
results = pd.DataFrame({
    'feature': selected_features,
    'coefficient': selected_coefs
}).sort_values('coefficient', key=np.abs, ascending=False)

results.head(20)

Total original features: 300000
Features after variance filter: 256190
Non-zero (selected) features: 234


,feature,coefficient
177,cg00932128,0.157652
134,cg15282417,-0.128633
186,cg05339390,0.094684
89,cg09872737,0.090768
41,cg18880500,-0.088677
108,cg18454120,0.085029
156,cg19067791,0.083734
189,cg12768969,0.082010
100,cg08931596,0.081819
192,cg16613233,0.080405


In [ ]:
# # Store the selected feature IDs and their coefficients

# # # Save feature IDs and coefficients to CSV
# results.to_csv('data/selected_DNAm_features.csv', index=False)

# print(f"Saved {len(selected_features)} selected features to:")
# print("  - data/selected_DNAm_features.csv (with coefficients)")

Saved 234 selected features to:
  - data/selected_DNAm_features.csv (with coefficients)


In [ ]:
# # Feature Selection using Lasso Regression

# # regress the gene expression features against the cigarettes per day
# # Basic penalized regression: gene expression features -> cigarettes_per_day

# # Response variable
# y = expMeta['cigarettes_per_day']

# # Drop samples with missing response
# valid_idx = y.dropna().index
# y = y.loc[valid_idx]

# # Align gene expression matrix rows to valid samples
# X = expData.loc[valid_idx]

# # Optional: remove features with too low variance (speeds up)
# var_filter = VarianceThreshold(threshold=0.01)  # adjust threshold as needed

# # Build pipeline: variance filter -> scaling -> LassoCV
# pipe = Pipeline([
#     ('var', var_filter),
#     ('scale', StandardScaler()),
#     ('lasso', LassoCV(cv=5, n_jobs=-1, max_iter=5000, verbose=False))
# ])

# # Fit model
# pipe.fit(X, y)

# lasso_ge = pipe.named_steps['lasso']
# filtered_feature_names_ge = X.columns[var_filter.get_support()]

# # Extract non-zero coefficients
# coef_ge = lasso_ge.coef_
# nonzero_mask_ge = coef_ge != 0
# selected_features_ge = filtered_feature_names_ge[nonzero_mask_ge]
# selected_coefs_ge = coef_ge[nonzero_mask_ge]

# # Report
# print(f"Total original features: {X.shape[1]}")
# print(f"Features after variance filter: {len(filtered_feature_names_ge)}")
# print(f"Non-zero (selected) features: {len(selected_features_ge)}")

# # Build results DataFrame
# results_ge = pd.DataFrame({
#     'feature': selected_features_ge,
#     'coefficient': selected_coefs_ge
# }).sort_values('coefficient', key=np.abs, ascending=False)

# # Display results
# print(f"\nTop 20 selected genes:")
# results_ge.head(20)

In [ ]:
# # Store the selected feature IDs and their coefficients

# # Save feature IDs and coefficients to CSV
# results_ge.to_csv('data/selected_GE_features.csv', index=False)

# print(f"Saved {len(selected_features_ge)} selected features to:")
# print("  - data/selected_DNAm_features.csv (with coefficients)")

In [ ]:
import pandas as pd
# subset the main data by these features to create the reduced size matrices from which we will build the PSNs

# load the selected features and subset the data
metFeatures = pd.read_csv('./data/selected_DNAm_features.csv')
geFeatures = pd.read_csv('./data/selected_GE_features.csv')

# subset metData by selected features
metData = metData[metFeatures['feature']]
metPatients,metFeatures = metData.shape
print(f'There are',metPatients,'patients with',metFeatures,'methylation sites selected')

# select expData by selected features
expData = expData[geFeatures['feature']]
expPatients,expFeatures = expData.shape
print(f'There are',expPatients,'patients with',expFeatures,'genes selected')

In [ ]:
# It's usually a good idea to us a dimensionality reduction method to look for outliers and distributions of samples
# Here we will do a basic PCA

# expression data

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# scale features, run PCA on 2-dimensions
X = expData.values
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
pcs = pca.fit_transform(X_scaled)
sample_pca = pd.DataFrame(pcs, columns=['PC1', 'PC2'], index=expData.index)

# add the sample_type so we can see if cancerous and non-cancerous samples are split
sample_pca = sample_pca.join(expMeta['sample_type'])
sample_pca.index.name = 'sample_id'
print(sample_pca)

In [ ]:
# plot the pca results
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

# map statuses to distinct colors
statuses = list(sample_pca['sample_type'].unique())
cmap = plt.get_cmap('Set1')
color_map = {s: cmap(i % cmap.N) for i, s in enumerate(statuses)}
colors = sample_pca['sample_type'].map(color_map)

# create figure and axes, scatter and attach colorbar to that axes
fig, ax = plt.subplots(figsize=(7,5))
sc = ax.scatter(sample_pca['PC1'], sample_pca['PC2'], s=50, edgecolor='k',c=list(colors))

# simple legend with colored patches
legend_elements = [Patch(facecolor=col, edgecolor='k', label=label) 
                   for label, col in color_map.items()]
ax.legend(handles=legend_elements, title='sample_type')

# labels and colorbar
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title('Sample Separation by Gene Expression Features')
plt.tight_layout()
plt.show()

In [ ]:
# It's usually a good idea to us a dimensionality reduction method to look for outliers and distributions of samples
# Here we will do a basic PCA

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# scale features, run PCA on 2-dimensions
X = metData.values
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
pcs = pca.fit_transform(X_scaled)
sample_pca = pd.DataFrame(pcs, columns=['PC1', 'PC2'], index=metData.index)

# add the sample_type so we can see if cancerous and non-cancerous samples are split
sample_pca = sample_pca.join(metMeta['sample_type'])
sample_pca.index.name = 'sample_id'
print(sample_pca)

In [ ]:
# plot the pca results
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

# map statuses to distinct colors
statuses = list(sample_pca['sample_type'].unique())
cmap = plt.get_cmap('Set1')
color_map = {s: cmap(i % cmap.N) for i, s in enumerate(statuses)}
colors = sample_pca['sample_type'].map(color_map)

# create figure and axes, scatter and attach colorbar to that axes
fig, ax = plt.subplots(figsize=(7,5))
sc = ax.scatter(sample_pca['PC1'], sample_pca['PC2'], s=50, edgecolor='k',c=list(colors))

# simple legend with colored patches
legend_elements = [Patch(facecolor=col, edgecolor='k', label=label) 
                   for label, col in color_map.items()]
ax.legend(handles=legend_elements, title='sample_type')

# labels and colorbar
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title('Sample Separation by DNA Methylation Features')
plt.tight_layout()
plt.show()

In [ ]:
# We will re-use some of the graph functions we worked with in our networks notebooks
import graph_functions as gf

# transform the gene expresison matrix so we can create a patient correlation matrix
pat_expData = expData.T

# create a dict() to hold our correlation matrices
correlation_matrices={}

# Pearson correlation - O(n^2) complexity, fast for large datasets
exp_corr = pat_expData.corr(method='pearson')

# add the matrix to our dict
correlation_matrices['expression'] = exp_corr

In [ ]:
# plot histogram of correlation values
import matplotlib.pyplot as plt
import numpy as np

# Extract upper triangle of correlation matrix (to avoid duplicates and diagonal)
upper_triangle_mask = np.triu(np.ones_like(exp_corr, dtype=bool), k=1)
correlation_values = exp_corr.where(upper_triangle_mask).values.flatten()
correlation_values = correlation_values[~np.isnan(correlation_values)]

# Plot histogram
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(correlation_values, bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Correlation Coefficient')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Gene Expression Pairwise Correlations')
ax.axvline(x=0, color='red', linestyle='--', linewidth=1, label='Zero correlation')
ax.legend()
plt.tight_layout()
plt.show()

# Print summary statistics
print(f"Mean correlation: {correlation_values.mean():.4f}")
print(f"Median correlation: {np.median(correlation_values):.4f}")
print(f"Std deviation: {correlation_values.std():.4f}")
print(f"Min correlation: {correlation_values.min():.4f}")
print(f"Max correlation: {correlation_values.max():.4f}")

In [ ]:
# repeat for our methylation data
pat_metData = metData.T

# Pearson correlation - O(n^2) complexity, fast for large datasets
met_corr = pat_metData.corr(method='pearson')

# add this to our correlation matrix dict
correlation_matrices['methylation'] = met_corr

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract upper triangle of correlation matrix (to avoid duplicates and diagonal)
upper_triangle_mask = np.triu(np.ones_like(met_corr, dtype=bool), k=1)
correlation_values = met_corr.where(upper_triangle_mask).values.flatten()
correlation_values = correlation_values[~np.isnan(correlation_values)]

# Plot histogram
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(correlation_values, bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Correlation Coefficient')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Gene Expression Pairwise Correlations')
ax.axvline(x=0, color='red', linestyle='--', linewidth=1, label='Zero correlation')
ax.legend()
plt.tight_layout()
plt.show()

# Print summary statistics
print(f"Mean correlation: {correlation_values.mean():.4f}")
print(f"Median correlation: {np.median(correlation_values):.4f}")
print(f"Std deviation: {correlation_values.std():.4f}")
print(f"Min correlation: {correlation_values.min():.4f}")
print(f"Max correlation: {correlation_values.max():.4f}")

In [ ]:
# plot the matrices side-by-side
gf.plot_correlation_matrices(correlation_matrices)

In [ ]:
# Create a graph from the Pearson correlation matrices with select thresholds (you could play with these, I've picked just above median)

# the PSN for expression data
patExpGraph = gf.create_graph_from_correlation(correlation_matrices['expression'], threshold=0.85)

# the PSN for methylation data
patMetGraph = gf.create_graph_from_correlation(correlation_matrices['methylation'], threshold=0.95)

In [ ]:
#add in missing patients from either graph (now we can weave in patients missing data from either modality)

# get the expression graph patient ids
expPatients = patExpGraph.nodes()
print(f'There are',len(expPatients),'patients in the expression graph')

# get the methylation graph patient ids
metPatients = patMetGraph.nodes()
print(f'There are',len(metPatients),'patients in the methylation graph')

#add nodes to methylation graph for patients that are in expression data but not in methylation
patMetGraph.add_nodes_from(list(set(expPatients)-set(metPatients)))

#add nodes to expression graph for patients that are in methylation data but not in expression
patExpGraph.add_nodes_from(list(set(metPatients)-set(expPatients)))

# get the expression graph patient ids
expPatients = patExpGraph.nodes()
print(f'There are now',len(expPatients),'patients in the expression graph')

# get the methylation graph patient ids
metPatients = patMetGraph.nodes()
print(f'There are now',len(metPatients),'patients in the methylation graph')

graphs = [patExpGraph,patMetGraph]

In [ ]:
# SNF fusion

# we need to make sure the nodes are in the same order in both graphs
nodes = sorted(patExpGraph.nodes())

# extract the adjacency matrices to perform SNF
A1 = nx.to_numpy_array(patExpGraph, nodelist=nodes, weight="weight")
A2 = nx.to_numpy_array(patMetGraph, nodelist=nodes, weight="weight")

# add the matrices to a list for the snf function
allgraphs = [A1,A2]

# perform SNF
affinity_networks = snf.make_affinity(allgraphs, metric='euclidean', K=20, mu=0.5)
# fuse the networks
fused_network = snf.snf(affinity_networks, K=20, t=20)

# remove self-loops
np.fill_diagonal(fused_network, 0)

In [ ]:
# This fused network will be completely connected (like a correlation matrix)
# We will use kNN to sparsify it

# retain only up to the 5 strongest edges per node
k = 5

# create an empty matrix to hold the edges we retain
A_knn = np.zeros_like(fused_network)

# perform the sparsification
for i in range(fused_network.shape[0]):
    row = fused_network[i].copy()
    row[i] = -np.inf
    idx = np.argsort(row)[-k:]
    A_knn[i, idx] = fused_network[i, idx]

# make sure the resulting matrix is symmetric
A_knn = np.maximum(A_knn, A_knn.T)

# create the graph object
G = nx.from_numpy_array(A_knn)

#add the patient ids back on
node_names = sorted(patExpGraph.nodes())
mapping = {i: node_name for i, node_name in enumerate(node_names)}
G = nx.relabel_nodes(G, mapping)


In [ ]:
# find the clustering coefficient of the fused graph
clustering_coeffs = nx.clustering(G)
avg_clustering_coeff = sum(clustering_coeffs.values()) / len(clustering_coeffs)
print(f"Average clustering coefficient for Fused Graph: {avg_clustering_coeff:.4f}")

plt.figure(figsize=(10, 10))

pos = nx.spring_layout(G, k=0.1, seed=50)
nx.draw_networkx_nodes(G, pos, node_size=200)
nx.draw_networkx_edges(G, pos, width=0.2, alpha=0.5)
plt.show()

In [ ]:
# Combine the MetaData (remembering that some patients were only present in one of the original data modalities)
import pandas as pd

# function to combine these data frames and collapse into unique columns
def merge_and_collapse(df1, df2, on):
    # Step 1: outer merge with temporary suffixes
    df_merged = pd.merge(df1, df2, on=on, how='outer', suffixes=('_1', '_2'))

    # Step 2: detect overlapping columns and combine
    cols_1 = [c for c in df_merged.columns if c.endswith('_1')]
    for col_1 in cols_1:
        base_col = col_1[:-2]
        col_2 = f'{base_col}_2'
        if col_2 in df_merged.columns:
            df_merged[base_col] = df_merged[col_1].combine_first(df_merged[col_2])
            df_merged = df_merged.drop(columns=[col_1, col_2])
        else:
            # if no _2 version, just rename _1 to base
            df_merged = df_merged.rename(columns={col_1: base_col})

    return df_merged

combinedMeta = merge_and_collapse(metMeta,expMeta,'patient')

In [ ]:
# A nice function that will take the fused network object and the metadata frame and allow us to plot with names and colour by attribute
def plot_snf_network(G, node_names, df, attr_col, cmap='tab10', node_size=100, with_labels=True):
    """
    Build a network from SNF adjacency matrix, relabel nodes with patient IDs,
    and plot colored by a categorical DataFrame attribute.

    Parameters:
    - A_knn: SNF adjacency matrix (numpy array)
    - node_names: list of node IDs in the same order as A_knn
    - df: DataFrame containing node attributes
    - attr_col: column in df to color nodes by
    - cmap: matplotlib colormap
    - node_size: size of nodes in plot
    - with_labels: whether to show labels
    """
    # Step 3: Prepare node attribute mapping
    df_sub = df.set_index('patient')
    node_attrs = df_sub[attr_col].to_dict()

    # Step 4: Generate colors for each category
    categories = list(sorted(set(node_attrs.values())))
    colormap = plt.get_cmap(cmap)
    n = len(categories)
    category_colors = {cat: matplotlib.colors.to_hex(colormap(i / max(1, n-1)))
                    for i, cat in enumerate(categories)}

    # Step 5: Assign node colors
    node_color_list = [category_colors.get(node_attrs.get(node, None), '#D3D3D3') 
                    for node in G.nodes()]

    # Step 6: Layout and draw
    pos = nx.spring_layout(G, seed=42, k=10.0)
    plt.figure(figsize=(10, 10))
    nx.draw(G, pos, node_color=node_color_list, with_labels=with_labels, node_size=node_size, edge_color='lightgray', font_size=8)

    # Step 7: Add legend
    legend_elements = [Patch(facecolor=color, label=cat) for cat, color in category_colors.items()]
    plt.legend(handles=legend_elements)
    plt.show()

    return G

In [ ]:
plot_snf_network(G,node_names,combinedMeta,'sample_type');

In [ ]:
# Cluster the gene correlation network using the greedy modularity communities algorithm

# Louvain with resolution
from community import community_louvain

partition = community_louvain.best_partition(G, resolution=0.9)
# Increase resolution (>1) -> more communities (finer)
# Decrease resolution (<1) -> fewer communities (coarser)

from collections import defaultdict

clusters = defaultdict(set)
for node, cluster_id in partition.items():
    clusters[cluster_id].add(node)
communities = list(clusters.values())

print(f'There are',len(communities),'communities')

In [ ]:
# Now we're going to plot the patient clusters with nodes colour coded by patient attributes

# Create a mapping from patient -> attribute
df_sub = combinedMeta.set_index('patient')
node_attrs = df_sub['sample_type'].to_dict()  # replace 'sample_type' with your attribute column
# node_attrs = df_sub['gender'].to_dict()  # replace 'sample_type' with your attribute column
# node_attrs = df_sub['race'].to_dict()  # replace 'sample_type' with your attribute column

# Get unique categories and assign colors
categories = sorted(set(node_attrs.values()))
colormap = plt.get_cmap('tab20')
category_colors = {cat: matplotlib.colors.to_hex(colormap(i / max(1, len(categories)-1)))
                   for i, cat in enumerate(categories)}

# Create subgraphs for each community
subgraphs = [G.subgraph(c) for c in communities]

for i, subgraph in enumerate(subgraphs):
    print(f'Community {i+1} has {subgraph.number_of_nodes()} nodes.')
    
    # Create list of node colors based on attribute
    node_color_list = [category_colors.get(node_attrs.get(node, None), '#D3D3D3')  # grey if missing
                       for node in subgraph.nodes()]
    
    plt.figure(figsize=(10, 10))
    pos = nx.spring_layout(subgraph, k=0.1, seed=50)
    nx.draw_networkx_nodes(subgraph, pos, node_size=200, node_color=node_color_list)
    nx.draw_networkx_edges(subgraph, pos, width=0.2, alpha=0.5)
    nx.draw_networkx_labels(subgraph, pos, font_size=10, font_color='black')
    
    # Optional: add legend for attribute colors
    legend_elements = [Patch(facecolor=color, label=cat) for cat, color in category_colors.items()]
    plt.legend(handles=legend_elements)
    
    plt.show()
